In [ ]:
#  ALANINE DIPEPTIDE 66D

!pip install -q mdtraj pot

import math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from scipy.stats import wasserstein_distance
import ot
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# LOAD DATA
PDB = "/content/alanine-dipeptide.pdb"
DCDS = [f"/content/trajectory{i}.dcd" for i in range(6)]
traj_list = [md.load(d, top=PDB) for d in DCDS]
traj = traj_list[0].join(traj_list[1:])
traj.superpose(traj, 0)

n_frames, n_atoms = traj.n_frames, traj.n_atoms
input_dim = n_atoms * 3
full_coords_real_nm = traj.xyz.copy().reshape(n_frames, -1)

mean = full_coords_real_nm.mean(axis=0)
std = full_coords_real_nm.std(axis=0) + 1e-8
full_coords_norm = (full_coords_real_nm - mean) / std

mean_t = torch.tensor(mean, dtype=torch.float32, device=device)
std_t = torch.tensor(std, dtype=torch.float32, device=device)

data_tensor = torch.tensor(full_coords_norm, dtype=torch.float32)
dataloader = DataLoader(TensorDataset(data_tensor), batch_size=256, shuffle=True)

# NONBONDED PAIRS
def build_graph_neighbors(topology):
    adj = {i: set() for i in range(topology.n_atoms)}
    for bond in topology.bonds:
        i, j = bond.atom1.index, bond.atom2.index
        adj[i].add(j); adj[j].add(i)
    return adj

def shortest_path_leq_2(adj, i, j):
    if j in adj[i]: return True
    for k in adj[i]:
        if j in adj[k]: return True
    return False

nonbonded_pairs = [(i,j) for i in range(traj.topology.n_atoms)
                   for j in range(i+1, traj.topology.n_atoms)
                   if not shortest_path_leq_2(build_graph_neighbors(traj.topology), i, j)]
print(f"Nonbonded pairs: {len(nonbonded_pairs)}")

# MODEL
class Coupling(nn.Module):
    def __init__(self, dim, hidden, mask):
        super().__init__()
        self.register_buffer("mask", mask)
        self.s_net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim), nn.Tanh()
        )
        self.t_net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim)
        )
    def forward(self, x):
        xm = x * self.mask
        s = 0.9 * self.s_net(xm) * (1 - self.mask)
        t = self.t_net(xm) * (1 - self.mask)
        y = xm + (1 - self.mask) * (x * torch.exp(s) + t)
        return y, torch.sum(s, dim=1)
    def inverse(self, y):
        ym = y * self.mask
        s = 0.9 * self.s_net(ym) * (1 - self.mask)
        t = self.t_net(ym) * (1 - self.mask)
        return ym + (1 - self.mask) * ((y - t) * torch.exp(-s))

class RealNVP(nn.Module):
    def __init__(self, dim, hidden=512, layers=12):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(layers):
            mask = torch.zeros(dim); mask[::2] = 1
            if i % 2 == 1: mask = 1 - mask
            self.layers.append(Coupling(dim, hidden, mask))
    def forward(self, x):
        ld = 0.0
        for l in self.layers:
            x, ldi = l(x)
            ld += ldi
        return x, ld
    def inverse(self, z):
        for l in reversed(self.layers):
            z = l.inverse(z)
        return z

def log_prob(z):
    d = z.shape[1]
    return -0.5 * torch.sum(z**2, dim=1) - 0.5 * d * math.log(2 * math.pi)

# PENALTY
def min_distance_penalty(x_phys, epsilon_reg=0.17):
    batch_size = x_phys.shape[0]
    coords_3d = x_phys.reshape(batch_size, n_atoms, 3)
    total_penalty = 0.0
    for (i, j) in nonbonded_pairs:
        diff = coords_3d[:, i, :] - coords_3d[:, j, :]
        r = torch.sqrt(torch.sum(diff**2, dim=1) + 1e-10)
        penalty = torch.clamp(epsilon_reg - r, min=0)
        total_penalty += torch.mean(penalty)
    return total_penalty / len(nonbonded_pairs)

# TRAINING FUNCTIONS
def train_without_reg(model, dataloader, epochs=200, lr=2e-4):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=30, factor=0.5)
    losses = []
    for ep in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in dataloader:
            x = batch[0].to(device)
            z, ld = model(x)
            loss = -torch.mean(log_prob(z) + ld)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        losses.append(avg_loss)
        scheduler.step(avg_loss)
        if (ep+1) % 50 == 0:
            print(f"  [Without Reg] Epoch {ep+1:3d}/{epochs} | Loss: {avg_loss:.4f}")
    return model, losses

def train_with_reg(model, dataloader, epochs=200, lr=2e-4, epsilon_reg=0.17, lambda_reg=5.0):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=30, factor=0.5)
    losses, nll_hist, reg_hist = [], [], []
    for ep in range(epochs):
        model.train()
        total_loss = 0.0
        total_nll = 0.0
        total_reg = 0.0
        for batch in dataloader:
            x = batch[0].to(device)
            z, ld = model(x)
            nll = -torch.mean(log_prob(z) + ld)
            z_rand = torch.randn(x.shape[0], input_dim, device=device)
            x_fake_norm = model.inverse(z_rand)
            x_fake_phys = x_fake_norm * std_t + mean_t
            reg = min_distance_penalty(x_fake_phys, epsilon_reg)
            loss = nll + lambda_reg * reg
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item()
            total_nll += nll.item()
            total_reg += reg.item()
        avg_loss = total_loss / len(dataloader)
        avg_nll = total_nll / len(dataloader)
        avg_reg = total_reg / len(dataloader)
        losses.append(avg_loss)
        nll_hist.append(avg_nll)
        reg_hist.append(avg_reg)
        scheduler.step(avg_loss)
        if (ep+1) % 50 == 0:
            print(f"  [With Reg] Epoch {ep+1:3d}/{epochs} | Loss: {avg_loss:.4f} | NLL: {avg_nll:.4f} | Reg: {avg_reg:.6f}")
    return model, losses, nll_hist, reg_hist

# TRAIN BOTH MODELS
print(f"\n Parameters: {sum(p.numel() for p in RealNVP(dim=input_dim).parameters()):,}")

print("\n TRAINING: WITHOUT REGULARIZATION")
model_wor, losses_wor = train_without_reg(RealNVP(dim=input_dim), dataloader, epochs=200)

print("\n TRAINING: WITH ε-REGULARIZATION (ε=0.17, λ=5.0, 200 epochs)")
model_wr, losses_wr, nll_wr, reg_wr = train_with_reg(RealNVP(dim=input_dim), dataloader, epochs=200, epsilon_reg=0.17, lambda_reg=5.0)

# GENERATE SAMPLES
print("\nGenerating samples...")
model_wor.eval(); model_wr.eval()
with torch.no_grad():
    z = torch.randn(10000, input_dim, device=device)
    gen_wor_norm = model_wor.inverse(z).cpu().numpy()
    gen_wr_norm = model_wr.inverse(z).cpu().numpy()
gen_wor_phys = gen_wor_norm * std + mean
gen_wr_phys = gen_wr_norm * std + mean

# METRICS
def compute_min_r(coords_nm, pairs, n_atoms, max_frames=5000):
    """minimum non‑bonded distance r"""
    coords_3d = coords_nm[:max_frames].reshape(-1, n_atoms, 3)
    r_min = []
    for frame in coords_3d:
        min_d = np.inf
        for i,j in pairs:
            d = np.linalg.norm(frame[i]-frame[j])
            if d < min_d: min_d = d
        r_min.append(min_d)
    return np.array(r_min)

def compute_lj_energy(coords_nm, pairs, n_atoms, sigma=0.17, epsilon=1.0, max_frames=5000):
    coords_3d = coords_nm[:max_frames].reshape(-1, n_atoms, 3)
    energies = []
    for frame in coords_3d:
        E = 0.0
        for i,j in pairs:
            r = np.linalg.norm(frame[i]-frame[j])
            if r < 0.001: E += 1e8
            else: E += 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)
        energies.append(E)
    return np.array(energies)

min_r_real = compute_min_r(full_coords_real_nm, nonbonded_pairs, n_atoms)
min_r_wor  = compute_min_r(gen_wor_phys, nonbonded_pairs, n_atoms)
min_r_wr   = compute_min_r(gen_wr_phys, nonbonded_pairs, n_atoms)

U_real = compute_lj_energy(full_coords_real_nm, nonbonded_pairs, n_atoms)
U_wor  = compute_lj_energy(gen_wor_phys, nonbonded_pairs, n_atoms)
U_wr   = compute_lj_energy(gen_wr_phys, nonbonded_pairs, n_atoms)

# PCA & W2
n_ot = 2000
idx_r  = np.random.choice(n_frames, n_ot, replace=False)
idx_wor = np.random.choice(len(gen_wor_norm), n_ot, replace=False)
idx_wr  = np.random.choice(len(gen_wr_norm), n_ot, replace=False)

pca = PCA(n_components=2)
real_pca = pca.fit_transform(full_coords_norm[idx_r])
wor_pca  = pca.transform(gen_wor_norm[idx_wor])
wr_pca   = pca.transform(gen_wr_norm[idx_wr])

M_wor = ot.dist(real_pca, wor_pca)
M_wr  = ot.dist(real_pca, wr_pca)
a = np.ones(n_ot)/n_ot
W2_wor = np.sqrt(ot.emd2(a, a, M_wor))
W2_wr  = np.sqrt(ot.emd2(a, a, M_wr))

# RESULTS TABLE
print("\n" + "="*20)
print("RESULTS: Without Reg vs With Reg")
print("="*20)
print(f"{'':<25} {'Real':>12} {'Without Reg':>14} {'With Reg':>12}")
print("-"*20)
print(f"{'min r [nm]':<25}")
print(f"{'  Minimum':<25} {min_r_real.min():>12.6f} {min_r_wor.min():>14.6f} {min_r_wr.min():>12.6f}")
print(f"{'  Mean':<25} {min_r_real.mean():>12.6f} {min_r_wor.mean():>14.6f} {min_r_wr.mean():>12.6f}")
print(f"{'U(t) [kJ/mol]':<25}")
print(f"{'  Maximum':<25} {U_real.max():>12.1f} {U_wor.max():>14.1e} {U_wr.max():>12.1f}")
print(f"{'  Mean':<25} {U_real.mean():>12.2f} {U_wor.mean():>14.2f} {U_wr.mean():>12.2f}")
print(f"{'PCA W2':<25} {'-':>12} {W2_wor:>14.4f} {W2_wr:>12.4f}")
print("="*20)

# PLOTS
plt.figure(figsize=(8,4))
plt.plot(losses_wor, 'r-', linewidth=1.5, label='Without Reg')
plt.plot(losses_wr, 'g-', linewidth=1.5, label='With Reg')
plt.xlabel("Epoch"); plt.ylabel("NLL Loss")
plt.title("Training Loss")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(real_pca[:,0], real_pca[:,1], s=3, alpha=0.3, c='blue')
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Real Distribution (PCA)")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(wor_pca[:,0], wor_pca[:,1], s=3, alpha=0.3, c='red')
plt.title("Generated - Without Reg (PCA)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(wr_pca[:,0], wr_pca[:,1], s=3, alpha=0.3, c='green')
plt.title("Generated - With Reg (PCA)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Distance histogram (log scale, named min r)
fig, ax = plt.subplots(figsize=(10,5))
bins = np.linspace(0.05, 0.30, 45)
ax.hist(min_r_real, bins=bins, alpha=0.6, density=True, color='blue',
        label=f'Real (min r = {min_r_real.min():.4f} nm)')
ax.hist(min_r_wor, bins=bins, alpha=0.5, density=True, color='red',
        label=f'Without Reg (min r = {min_r_wor.min():.4f} nm)')
ax.hist(min_r_wr, bins=bins, alpha=0.5, density=True, color='green',
        label=f'With Reg (min r = {min_r_wr.min():.4f} nm)')
ax.set_xlabel("min r = minimum nonbonded distance [nm]")
ax.set_ylabel("Density (log scale)")
ax.set_title("Minimum Atomic Distance (log scale)")
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# STATIC 3-PANEL PHI/PSI PLOT

print("\nGenerating Static 3-Panel Phi/Psi Plot")

# 1. Generate Phi/Psi for Real MD
phi_real_rad = md.compute_phi(traj)[1]
psi_real_rad = md.compute_psi(traj)[1]
phi_real = np.degrees(phi_real_rad).flatten()
psi_real = np.degrees(psi_real_rad).flatten()

# 2. Generate Phi/Psi for the Without Reg model
model_wor.eval()
n_samples_static = 20000
with torch.no_grad():
    z_wor = torch.randn(n_samples_static, input_dim, device=device)
    x_wor_norm = model_wor.inverse(z_wor).cpu().numpy()
    x_wor_phys = x_wor_norm * std + mean

coords_wor = x_wor_phys.reshape(-1, n_atoms, 3)
traj_wor = md.Trajectory(coords_wor, traj.topology)
phi_wor = np.degrees(md.compute_phi(traj_wor)[1]).flatten()
psi_wor = np.degrees(md.compute_psi(traj_wor)[1]).flatten()

# 3. Generate Phi/Psi for the With Reg model
model_wr.eval()
with torch.no_grad():
    z_wr = torch.randn(n_samples_static, input_dim, device=device)
    x_wr_norm = model_wr.inverse(z_wr).cpu().numpy()
    x_wr_phys = x_wr_norm * std + mean

coords_wr = x_wr_phys.reshape(-1, n_atoms, 3)
traj_wr = md.Trajectory(coords_wr, traj.topology)
phi_wr = np.degrees(md.compute_phi(traj_wr)[1]).flatten()
psi_wr = np.degrees(md.compute_psi(traj_wr)[1]).flatten()

# 4. Plot the 3 panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Real MD
ax = axes[0]
real_idx = np.random.choice(len(phi_real), n_samples_static, replace=False)
ax.scatter(phi_real[real_idx], psi_real[real_idx], s=3, alpha=0.3, c='blue')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel('(Phi) (degrees)'); ax.set_ylabel('(Psi) (degrees)')
ax.set_title(r'(a) Real Distribution (Phi, Psi)', fontsize=12)
ax.grid(True, alpha=0.3)

# Panel 2: Generated  Without Reg
ax = axes[1]
ax.scatter(phi_wor, psi_wor, s=3, alpha=0.3, c='red')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel('(Phi) (degrees)'); ax.set_ylabel('(Psi) (degrees)')
ax.set_title('(b) Generated distribution(Phi, Psi)  Without Regularization', fontsize=12)
ax.grid(True, alpha=0.3)

# Panel 3: Generated With Reg
ax = axes[2]
ax.scatter(phi_wr, psi_wr, s=3, alpha=0.3, c='green')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel('(Phi) (degrees)'); ax.set_ylabel('(Psi) (degrees)')
ax.set_title('(c) Generated distribution (Phi,Psi)  With Regularization', fontsize=12)
ax.grid(True, alpha=0.3)

plt.suptitle(r" Phi-Psi from 66D Cartesian Coordinate space", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
plt.savefig('phipsi_3panel_static.pdf', dpi=300, bbox_inches='tight')




# 3D ANIMATION

print("\nGenerating Final 3D Animation ")
model_wr.eval()

n_samples = 3000

# GENERATE DATA
with torch.no_grad():
    z_prior = torch.randn(n_samples, input_dim, device=device)
    x_mapped_norm = model_wr.inverse(z_prior).cpu().numpy()
    x_mapped_phys = x_mapped_norm * std + mean

# Right Panel Data (Phi/Psi)
coords_generated = x_mapped_phys.reshape(-1, n_atoms, 3)
traj_gen = md.Trajectory(coords_generated, traj.topology)
phi_gen_end = np.degrees(md.compute_phi(traj_gen)[1]).flatten()
psi_gen_end = np.degrees(md.compute_psi(traj_gen)[1]).flatten()

# Left Panel Data (1 molecule for 3D)
z_prior_mol = torch.randn(1, input_dim, device=device)
x_mapped_mol = model_wr.inverse(z_prior_mol).detach().cpu().numpy() * std + mean
mol_gen_end = x_mapped_mol.reshape(n_atoms, 3)

# Start States
phi_gaussian_start = np.random.uniform(-180, 180, n_samples)
psi_gaussian_start = np.random.uniform(-180, 180, n_samples)
mol_gaussian_start = np.random.normal(0, 0.3, (n_atoms, 3))

# Real MD Target
real_idx = np.random.randint(0, len(full_coords_real_nm))
real_md_frame = full_coords_real_nm[real_idx].reshape(n_atoms, 3)


# CENTER

def center_molecule(coords):
    return coords - np.mean(coords, axis=0)

real_md_frame = center_molecule(real_md_frame)
mol_gen_end = center_molecule(mol_gen_end)

def kabsch_align(source, target):
    H = np.dot(source.T, target)
    U, S, Vt = np.linalg.svd(H)
    R = np.dot(Vt.T, U.T)
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = np.dot(Vt.T, U.T)
    return np.dot(source, R)

mol_gen_end = kabsch_align(mol_gen_end, real_md_frame)

# FIGURE
fig = plt.figure(figsize=(14, 8))

# Panel 1: Left (3D Molecule)
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.set_xlim(-0.5, 0.5); ax1.set_ylim(-0.5, 0.5); ax1.set_zlim(-0.5, 0.5)
ax1.set_xlabel('X (nm)'); ax1.set_ylabel('Y (nm)'); ax1.set_zlabel('Z (nm)')
ax1.set_title('Alanine dipeptide molecule positions of 22 atoms (66 coordinates)', fontsize=14)
ax1.grid(True, alpha=0.2)

# Real MD Target (Blue)
blue_dots = ax1.scatter(real_md_frame[:,0], real_md_frame[:,1], real_md_frame[:,2], c='blue', s=45, alpha=0.5, label='Real Boltzmann sample')
blue_lines = []
for bond in traj.topology.bonds:
    line, = ax1.plot([], [], [], c='blue', alpha=0.3, linewidth=2.0)
    blue_lines.append(line)

# Generated Molecule (Green)
mol_lines = []
for bond in traj.topology.bonds:
    line, = ax1.plot([], [], [], c='green', alpha=0.6, linewidth=2.5)
    mol_lines.append(line)
scat_mol = ax1.scatter([], [], [], c='green', s=55, alpha=0.8, label='Generated flow sample')
ax1.legend(loc='upper right')

# Panel 2: Right (2D Phi/Psi)
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_xlim(-180, 180); ax2.set_ylim(-180, 180)
ax2.set_xlabel('Phi(degrees)'); ax2.set_ylabel('(Psi) (degrees)')
ax2.set_title('2D projection of 66D distribution (Phi, Psi)', fontsize=14)
ax2.grid(True, alpha=0.2)

n_real_show = 3000
real_indices = np.random.choice(len(phi_real), n_real_show, replace=False)
ax2.scatter(phi_real[real_indices], psi_real[real_indices], s=3, alpha=0.4, c='blue', label='Real Boltzmann distribution')
scat_phipsi = ax2.scatter([], [], s=8, alpha=0.7, c='green', label='Generated flow')
ax2.legend(loc='upper right')

# ANIMATION UPDATE
n_frames = 30
total_layers = 12

# Calculate frames
blue_curr_frames = []
green_curr_frames = []
for i in range(n_frames):
    t = i / (n_frames - 1)

    g = (1 - t) * mol_gaussian_start + t * mol_gen_end
    g = center_molecule(g)

    b = real_md_frame.copy()
    b = center_molecule(b)


    if t > 0.99:
        g = b.copy()

    green_curr_frames.append(g)
    blue_curr_frames.append(b)
def update(frame_idx):
    t = frame_idx / (n_frames - 1)

    # Right Panel
    phi_curr = (1 - t) * phi_gaussian_start + t * phi_gen_end
    psi_curr = (1 - t) * psi_gaussian_start + t * psi_gen_end
    scat_phipsi.set_offsets(np.c_[phi_curr, psi_curr])

    # Left Panel
    mol_curr = green_curr_frames[frame_idx]
    blue_curr = blue_curr_frames[frame_idx]

    # Update
    scat_mol._offsets3d = (mol_curr[:,0], mol_curr[:,1], mol_curr[:,2])
    for i, bond in enumerate(traj.topology.bonds):
        idx1, idx2 = bond.atom1.index, bond.atom2.index
        mol_lines[i].set_data([mol_curr[idx1,0], mol_curr[idx2,0]], [mol_curr[idx1,1], mol_curr[idx2,1]])
        mol_lines[i].set_3d_properties([mol_curr[idx1,2], mol_curr[idx2,2]])

    # Update
    blue_dots._offsets3d = (blue_curr[:,0], blue_curr[:,1], blue_curr[:,2])
    for i, bond in enumerate(traj.topology.bonds):
        idx1, idx2 = bond.atom1.index, bond.atom2.index
        blue_lines[i].set_data([blue_curr[idx1,0], blue_curr[idx2,0]], [blue_curr[idx1,1], blue_curr[idx2,1]])
        blue_lines[i].set_3d_properties([blue_curr[idx1,2], blue_curr[idx2,2]])

    # Titles
    layer_num = int(round(t * total_layers))
    if layer_num == 0:
        status = "Layer 0: Gaussian Prior"
    else:
        status = f"After Layer {layer_num} / {total_layers}"

    ax1.set_title(f'Alanine dipeptide molecule\npositions of 22 atoms (66 coordinates): {status}', fontsize=12)
    ax2.set_title(f'2D projection of 66D distribution: {status}', fontsize=12)

    return [scat_mol, scat_phipsi, blue_dots] + mol_lines + blue_lines

# RENDER

ani = FuncAnimation(fig, update, frames=n_frames, interval=1000, blit=False)
ani.save('66d_3d_layer_by_layer_animation.mp4', writer='ffmpeg', fps=1)


plt.close()
HTML(ani.to_jshtml())
